# Readable colour bars for `sym_log` and `log` (cleopatra #335)

Before the fix, a non-linear colour scale left the bar effectively unlabelled (~1 of 11 ticks). This notebook shows the fixed behaviour: **decade-aligned, signed tick labels**, and how the ticks are *spaced*.

> Requires cleopatra with the #335 fix (`ColorScaling.build_norm` scale-aware ticks).

In [ ]:
%matplotlib inline
import numpy as np
from cleopatra.glyphs.gridded.array_glyph import ArrayGlyph
from cleopatra.styling.scaling import ColorScaling

# Terrain-like: most cells near sea level, a depression, a long tail to ~744 m
rng = np.random.default_rng(0)
values = np.concatenate([
    rng.normal(4.0, 3.0, 60_000),   # the plain, 0-10 m
    rng.uniform(-24.0, 0.0, 8_000), # below sea level
    rng.uniform(10.0, 50.0, 20_000),
    rng.uniform(50.0, 744.0, 12_000),
]).reshape(200, 500)
values.min().round(1), values.max().round(1)

## `sym_log` — signed, spans below and above zero

Ticks land on the symlog **decades** within the range and keep their sign.

In [ ]:
g = ArrayGlyph(values)
fig, ax = g.plot(cmap='terrain', color=ColorScaling.sym_log(threshold=10.0, scale=1.0))
print('tick labels:', [t.get_text() for t in g.cbar.ax.get_yticklabels()])
fig

## `log` — strictly positive

Ticks land on the log **decades** (1, 10, 100, ...).

In [ ]:
gl = ArrayGlyph(np.abs(values) + 0.5)   # strictly positive so log() is legal
figl, axl = gl.plot(cmap='terrain', color=ColorScaling.log())
print('tick labels:', [t.get_text() for t in gl.cbar.ax.get_yticklabels()])
figl

## Your own ticks are labelled as asked

`cbar.set_ticks([...])` now labels the positions directly — no paired `set_ticklabels()` needed (that was the second half of #335).

In [ ]:
g2 = ArrayGlyph(values)
fig2, ax2 = g2.plot(cmap='terrain', color=ColorScaling.sym_log(threshold=10.0, scale=1.0))
g2.cbar.set_ticks([-20, -5, 0, 5, 10, 20, 50, 100, 300, 700])
print('tick labels:', [t.get_text() for t in g2.cbar.ax.get_yticklabels()])
fig2

## How the ticks are *spaced*

The **labels are data values**, but each tick's physical position on the bar follows the norm's transform, not even data-spacing:

- **log** — every decade (×10) is the *same* physical distance: `1`, `10`, `100` are evenly spaced.
- **sym_log** — a **linear band** for |value| <= `threshold` (here `-10 . 0 . 10` are evenly spaced) and **log-compressed wings** beyond it (`10 -> 100` is one decade). So values near zero get more bar space than values out in the tail — which is why sym_log suits data clustered near zero with a long tail.

The cell below prints each tick's normalised (0..1) bar position to make the spacing explicit.

In [ ]:
for name, cs, data in [
    ('sym_log', ColorScaling.sym_log(threshold=10.0, scale=1.0), values),
    ('log',     ColorScaling.log(),                              np.abs(values) + 0.5),
]:
    gg = ArrayGlyph(data)
    gg.plot(cmap='terrain', color=cs)
    norm = gg.im.norm
    ticks = gg.cbar.get_ticks()
    print(name, '->')
    for t in ticks:
        print(f'   tick {t:8.2f}   bar position {float(norm(t)):.3f}')